In [1]:
import warnings
import duckdb
import numpy as np
import torch
from torch import nn, optim
from tqdm import tqdm

pairs = (
  'BTCUSDT',
  'ETHUSDT',
  'SOLUSDT',
  'BNBUSDT'
)

n_pairs = len(pairs)
print(f'Pairs: {n_pairs}')

index = np.arange(
  np.datetime64('2021-07-01'),
  np.datetime64('2026-07-01'),
  np.timedelta64(1, 'm')
)

n_obs = len(index)
print(f'Observations: {n_obs}')

ohlc = {}
for pair in pairs:
  ohlc[pair] = duckdb.read_parquet(
    f'../data/processed/{pair}_2021_2026.parquet'
  ).fetchnumpy()

Pairs: 4
Observations: 2629440


In [2]:
class Model(nn.Module):
  def __init__(self, d_input: int, d_hidden: int, d_output: int, n_layers: int):
    super().__init__()
    self.gru = nn.GRU(d_input, d_hidden, n_layers, batch_first=True)
    self.fc = nn.Linear(d_hidden, d_output)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.fc(self.gru(x)[1][0])

model = Model(9, 90, 2, 2)
optimizer = optim.NAdam(model.parameters(), 1e-6)

n_params = sum(param.numel() for param in model.parameters())
print(n_params)

76592


In [3]:
min_tf = 180
max_tf = 720
min_frames = 25
max_frames = 25

maker_fee = 0.0002
taker_fee = 0.0005
deposit = 100.0

batch_size = 50
n_batches = 50
n_epochs = 10

for epoch in range(n_epochs):

  model.train()
  total_gain, total_wins = 0.0, 0.0
  pbar = tqdm(range(1, n_batches + 1), f'epoch={epoch} training')
  for n in pbar:

    optimizer.zero_grad()
    pnls = []
    for _ in range(batch_size):

      pair = np.random.choice(pairs)
      patience = tf = np.random.randint(min_tf, max_tf + 1)
      n_frames = np.random.randint(min_frames, max_frames + 1)
      lookback = (n_frames + 1) * tf
      t0 = np.random.randint(lookback - 1, n_obs - patience - 1)

      sliced = {
        k: v[t0 + 1 - lookback : t0 + 1].reshape(n_frames + 1, tf) 
        for k, v in ohlc[pair].items()
      }

      o = sliced['open'][:, 0]
      h = sliced['high'].max(axis=1)
      l = sliced['low'].min(axis=1)
      c = sliced['close'][:, -1]
      
      v = sliced['volume'].sum(1)
      qav = sliced['qav'].sum(1)
      nt = sliced['n_trades'].sum(1)
      bv = sliced['buy_volume'].sum(1)
      bqav = sliced['buy_qav'].sum(1)

      with warnings.catch_warnings():

        tmp00 = v - bv
        tmp01 = qav - bqav
        tmp02 = bv / bqav
        tmp03 = tmp00 / tmp01

        tmp04 = bqav / tmp01
        tmp05 = bv / tmp00

        tmp06 = o * tmp02
        tmp07 = o * tmp03

        tmp08 = h * tmp02
        tmp09 = l * tmp03

        tmp10 = c * tmp02
        tmp11 = c * tmp03

        tmp12 = np.where(
          c > o,
          (c * l) / np.square(o),
          (o * l) / np.square(c)
        )
        tmp13 = bv[1:] / bv[:-1]
        tmp14 = bqav[1:] / bqav[:-1]
        tmp15 = nt[1:] / nt[:-1]

        src = np.log(np.stack((
          tmp04[1:],
          (tmp05 / tmp04)[1:],
          (tmp06 * tmp07)[1:],
          (tmp08 * tmp09)[1:],
          (tmp10 * tmp11)[1:],
          tmp12[1:],
          tmp13,
          tmp13 / tmp14,
          tmp13 / tmp15
        ), 1))

        src = np.nan_to_num(src, nan=0.0, posinf=0.0, neginf=0.0)

      x = torch.from_numpy(src).float().unsqueeze(0)
      y = model(x).squeeze(0)
      bprice, sprice = y.exp() * c[-1]

      qty, spent, taken = deposit / c[-1], 0.0, 0.0
      for t in range(t0 + 1, n_obs):
        span = t - t0 - 1
        o, h, l, c = (ohlc[pair][key][t] for key in ('open', 'high', 'low', 'close'))
        if span == patience:
          if not spent: spent = bprice.detach() / bprice * o * (1.0 + taker_fee) * qty
          if not taken: taken = sprice.detach() / sprice * o * (1.0 - taker_fee) * qty
        else:
          if not spent and l <= bprice: spent = (
            bprice / bprice.detach() * o * (1.0 + taker_fee)
            if not span and o <= bprice else bprice * (1.0 + maker_fee)
          ) * qty
          if not taken and h >= sprice: taken = (
            sprice / sprice.detach() * o * (1.0 - taker_fee)
            if not span and o >= sprice else sprice * (1.0 - maker_fee)
          ) * qty
        if spent and taken:
          pnls.append(taken - spent)
          break

    pnls = torch.stack(pnls)
    gain = pnls.mean()
    (-gain).backward()
    optimizer.step()

    total_gain += gain.item()
    total_wins += (pnls > 0.0).float().mean().item()
    pbar.set_postfix_str(f'avg_pnl={total_gain/n:.4f}, win_rate={total_wins/n:.4f}')

epoch=9 training: 100%|██████████| 50/50 [01:21<00:00,  1.62s/it, avg_pnl=-0.0263, win_rate=0.4780]
